# Thí nghiệm 1: Phân tích Ước lượng (Estimation)


In [ ]:
import os, sys, requests
if 'google.colab' in sys.modules:
    data_dir = '/content/data'
else:
    data_dir = os.path.abspath('../data') if os.path.exists('../data') else os.path.abspath('data')
os.makedirs(data_dir, exist_ok=True)
path = os.path.join(data_dir, 'insurance.csv')
if not os.path.exists(path):
    r = requests.get('https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv')
    with open(path, 'wb') as f: f.write(r.content)
print(f'✅ Dữ liệu sẵn sàng tại: {path}')


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
def run_full_estimation(data, stat_func, n_boot=2000):
    orig = stat_func(data)
    boot_dist = np.array([stat_func(np.random.choice(data, len(data), replace=True)) for _ in range(n_boot)])
    n = len(data)
    jk_dist = np.array([stat_func(np.delete(data, i)) for i in range(n)])
    jk_se = np.sqrt(((n-1)/n) * np.sum((jk_dist - np.mean(jk_dist))**2))
    return {'boot_se': np.std(boot_dist), 'jk_se': jk_se}


In [ ]:
import os
data_dir = '/content/data' if 'google.colab' in os.sys.modules else (os.path.abspath('../data') if os.path.exists('../data') else os.path.abspath('data'))
df = pd.read_csv(os.path.join(data_dir, 'insurance.csv'))
data = df['charges'].sample(50, random_state=1).values
r_mean = run_full_estimation(data, np.mean)
r_med = run_full_estimation(data, np.median)

sns.set_theme(style='whitegrid', context='talk')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Biểu đồ MEAN
ax1.bar(['Bootstrap', 'Jackknife'], [r_mean['boot_se'], r_mean['jk_se']], color=['#3498db', '#e67e22'])
ax1.set_title('Ước lượng sai số cho MEAN', fontsize=14)
ax1.set_xlabel('Phương pháp Resampling')
ax1.set_ylabel('Standard Error (Đơn vị: USD)')

# Biểu đồ MEDIAN
ax2.bar(['Bootstrap', 'Jackknife'], [r_med['boot_se'], r_med['jk_se']], color=['#3498db', '#e74c3c'])
ax2.set_title('Ước lượng sai số cho MEDIAN', fontsize=14)
ax2.set_xlabel('Phương pháp Resampling')
ax2.set_ylabel('Standard Error (Đơn vị: USD)')

plt.tight_layout()
plt.show()
